# 🧠 Enpresetan Erabiltzen Diren 9 Machine Learning Ereduak


**Fikziozko enpresa** bat sortuko dugu (online denda bat, bezero, erosketa eta iruzkinekin) eta klasean azaldutako 9 ereduak aplikatuko dizkiogu, bata bestearen atzetik — hasieratik amaierara **bezero berberen gainean**.

---

## ▶️ Setup

1. Beheko moduko kode-blokeak ikusiko dituzu. Egin klik ezkerreko **▶** botoian exekutatzeko — ez da beharrezkoa kodea guztiz ulertzea.
2. *"Google-k egiaztatu gabea"* dioen abisua agertzen bazaizu, normala da — sakatu **"Exekutatu hala ere"**.
3. Joan behera eta sakatu ▶ bloke bakoitzean, ordenan. Bakoitzak segundo gutxi batzuk behar ditu.

**Ez duzu gelaxkaz gelaxka joan nahi?** Joan hona: `Exekuzio-ingurunea → Exekutatu guztiak` (`Runtime → Run all`), eta itxaron pare bat minutu bere kabuz amaitu arte.

Ez duzu ezer instalatu behar. Dena hodeian exekutatzen da, zure Google kontuan bertan.

---

In [ ]:
# Erabiliko ditugun liburutegiak instalatu/eguneratu (segundo batzuk)
!pip install -q -U scikit-learn xgboost lightgbm ipywidgets

## 🏪 Eszenatokia prestatzen

Gure fikziozko enpresa **Uni Eibar Market** da, bezero errepikakorrak dituen online denda bat. 2.000 bezeroko datu-multzo sintetiko bat sortuko dugu — haien gastuarekin, erosketa-maiztasunarekin, laguntza-intzidentziarik izan duten ala ez, eta utzi duten azken iruzkinarekin.

Datu-multzo hau hemen bertan sortzen da kode bidez; zuk ez duzu inolako datu-multzorik kargatu behar.

Eta bezero jakin bati jarraituko diogu —gure **Izar Bezeroari**— 9 ereduetan zehar, bakoitzak hari buruz zer dioen ikusteko.

Kode honek zaila eman dezake, baina ez da beharrezkoa xehetasunez ulertzea, datuak sortzeko zatia besterik ez baita.

Benetako egoera batean datuak lehendik izango zenituzke eta inportatzea askoz errazagoa litzateke.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
np.set_printoptions(suppress=True)


def generate_dataset(n=2000, seed=42):
    rng = np.random.default_rng(seed)

    age = rng.normal(40, 12, n).clip(18, 75).round().astype(int)
    tenure_months = rng.integers(1, 61, n)
    base_spend = rng.gamma(shape=4.0, scale=35, size=n) + 20
    monthly_purchases = rng.poisson(lam=(base_spend / 60)).clip(0, 20)
    monthly_email_opens = rng.poisson(lam=3.2, size=n).clip(0, 15)
    days_since_last_purchase = rng.exponential(scale=18, size=n).clip(0, 180).round().astype(int)
    support_ticket = rng.binomial(1, 0.12, n)

    z_churn = (
        -0.8
        + 0.022 * days_since_last_purchase
        - 0.15 * monthly_email_opens
        - 0.012 * tenure_months
        + 0.70 * support_ticket
        - 0.003 * base_spend
    )
    p_churn = 1 / (1 + np.exp(-z_churn))
    churn = rng.binomial(1, p_churn)

    # Nahita EZ-lineala den erlazioa: deskonexio-atalasea + gastu handiko bezero
    # maizentzat bonusa + U alderantzikatuko forma antzinatasunean.
    # Erregresio lineal batek ezin ditu ondo jaso efektu hauek; zuhaitz-eredu batek bai.
    factor = 0.92 + 0.002 * tenure_months - 0.35 * churn
    factor = factor - 0.28 * (days_since_last_purchase > 60)
    factor = factor + 0.18 * ((monthly_purchases >= 5) & (base_spend > 150))
    factor = factor - 0.00035 * (tenure_months - 28) ** 2
    noise = rng.normal(0, 10, n)
    next_monthly_spend = (base_spend * factor + noise).clip(0, None).round(2)

    z_camp = (
        -1.4
        + 0.012 * base_spend
        + 0.15 * monthly_purchases
        + 0.01 * tenure_months
        - 0.02 * days_since_last_purchase
    )
    p_camp = 1 / (1 + np.exp(-z_camp))
    responded_campaign = rng.binomial(1, p_camp)

    categories = np.array(["Elektronika", "Moda", "Etxea", "Kirola", "Edertasuna"])
    weights_by_age = np.stack([
        np.clip(0.35 - 0.004 * (age - 25), 0.05, 0.6),
        np.full(n, 0.22),
        np.clip(0.15 + 0.004 * (age - 25), 0.1, 0.4),
        np.full(n, 0.18),
        np.full(n, 0.15),
    ], axis=1)
    weights_by_age = weights_by_age / weights_by_age.sum(axis=1, keepdims=True)
    favorite_category = np.array([
        rng.choice(categories, p=weights_by_age[i]) for i in range(n)
    ])

    return pd.DataFrame({
        "customer_id": np.arange(1, n + 1),
        "age": age,
        "tenure_months": tenure_months,
        "current_monthly_spend": base_spend.round(2),
        "monthly_purchases": monthly_purchases,
        "days_since_last_purchase": days_since_last_purchase,
        "monthly_email_opens": monthly_email_opens,
        "support_ticket": support_ticket,
        "favorite_category": favorite_category,
        "churn": churn,
        "next_monthly_spend": next_monthly_spend,
        "responded_campaign": responded_campaign,
    })


def generate_reviews(n=2000, seed=7):
    rng = np.random.default_rng(seed)
    positives = [
        "oso pozik zerbitzuarekin", "dena garaiz eta egoera onean iritsi zen",
        "bezeroarentzako arreta bikaina", "nire zalantza oso azkar argitu zuten",
        "produktuak nire itxaropenak gainditu zituen", "bidalketa azkarra eta arazorik gabe",
        "aplikazioa oso erraza da erabiltzeko", "kalitate-prezio erlazio bikaina",
        "laguntza-taldea oso atsegina izan zen", "deskribatutako guztia bezalaxe",
        "zalantzarik gabe berriro erosiko dut", "esperientzia zoragarria hasieratik amaierara",
    ]
    negatives = [
        "eskaera berandu iritsi zen", "oso etsita kalitatearekin",
        "arreta-zerbitzuak ez du erantzuten", "inork ez zidan lagundu arazoa konpontzen",
        "aplikazioa etengabe blokeatzen da", "ez dut hemen berriro erosteko asmorik",
        "produktua hondatuta iritsi zen", "gehiago kobratu zidaten azalpenik gabe",
        "esperientzia oso txarra oro har", "laguntzak egunak behar izan zituen erantzuteko",
        "esperientzia nahiko frustragarria", "itzultze-prozesua oso motela da",
    ]
    texts, labels = [], []
    for _ in range(n):
        is_positive = rng.random() < 0.5
        bank = positives if is_positive else negatives
        phrases = rng.choice(bank, size=rng.integers(1, 3), replace=False)
        if rng.random() < 0.06:
            phrases = np.append(phrases, rng.choice(negatives if is_positive else positives, size=1))
        text = ". ".join(p.capitalize() for p in phrases) + "."
        texts.append(text)
        labels.append(1 if is_positive else 0)
    return pd.DataFrame({"review_text": texts, "positive_sentiment": labels})


def get_star_customer():
    # Fikziozko bezeroa, entrenamenduan parte hartzen EZ duena — haren gainean soilik aurreikusiko dugu.
    return pd.DataFrame([{
        "customer_id": 99999,
        "age": 34,
        "tenure_months": 14,
        "current_monthly_spend": 145.0,
        "monthly_purchases": 2,
        "days_since_last_purchase": 46,
        "monthly_email_opens": 1,
        "support_ticket": 1,
    }])


def print_card(title, lines):
    print("=" * 58)
    print(title)
    print("=" * 58)
    for l in lines:
        print(" •", l)
    print()


df = generate_dataset()
reviews = generate_reviews()
star_cust = get_star_customer()

print(f"Sortutako datu-multzoa: {len(df):,} Uni Eibar Market-eko bezero\n")
df.head()

In [ ]:
print_card("👤 BEZERO IZARRA", [
    f"ID: {star_cust.customer_id.iloc[0]}",
    f"Adina: {star_cust.age.iloc[0]} urte",
    f"Bezeroa duela: {star_cust.tenure_months.iloc[0]} hilabete",
    f"Hileko egungo gastua: {star_cust.current_monthly_spend.iloc[0]}€",
    f"Erosketak azken hilabetean: {star_cust.monthly_purchases.iloc[0]}",
    f"Azken erosketatik igarotako egunak: {star_cust.days_since_last_purchase.iloc[0]}",
    f"Azken hilabetean irekitako emailak: {star_cust.monthly_email_opens.iloc[0]}",
    f"Laguntza-intzidentziarik izan al zuen?: {'Bai' if star_cust.support_ticket.iloc[0] else 'Ez'}",
])
print("Bezero hau 9 ereduetatik pasatuko dugu eta bakoitzak hari buruz zer dioen ikusiko dugu.")

Hemendik aurrera algoritmo bakoitzaren kodea hasten da.

Helburua analisiaren egitura barneratzea da.

Erreparatzen badiozu, algoritmo zehatza edozein dela ere, scikit-learn erabiliz ia beti egitura bera jarraitzen da:

*   Erabili beharreko algoritmoa inportatu
*   Algoritmo horren instantzia bat sortu aldagai batean
*   `fit` metodoarekin entrenatu
*   `predict` edo `predict_proba` bidez aurreikusi

Jakina, hau oinarrizko hezurdura besterik ez da, ohitzen has zaitezen.

---
## 1️⃣ Erregresio Lineala — "Zenbat gastatuko du?"

**Zenbaki** bat aurreikusten du: bezeroak hurrengo hilabetean egingo duen gastua, egungo gastutik eta antzinatasunetik abiatuta.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

linear_features = ["current_monthly_spend", "tenure_months"]
X_train, X_test, y_train, y_test = train_test_split(
    df[linear_features], df["next_monthly_spend"], test_size=0.2, random_state=42
)

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

pred_linear = linear_model.predict(star_cust[linear_features])[0]

print_card("📈 ERREGRESIO LINEALA", [
    f"Ereduaren zehaztasuna (R²) inoiz ikusi ez dituen datuetan: {linear_model.score(X_test, y_test):.2f}",
    f"Gure izar bezeroarentzat hurrengo hilabeteko gastuaren aurreikuspena: {pred_linear:.2f}€",
])

---
## 2️⃣ Erregresio Logistikoa — "Baja emango al du?"

**0 eta 1 arteko probabilitate** bat aurreikusten du: bezero honek baja emateko (churn) probabilitatea.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

logistic_features = ["days_since_last_purchase", "monthly_email_opens", "tenure_months",
                     "support_ticket", "current_monthly_spend"]
X_train, X_test, y_train, y_test = train_test_split(
    df[logistic_features], df["churn"], test_size=0.2, random_state=42, stratify=df["churn"]
)

logistic_model = LogisticRegression()
logistic_model.fit(X_train, y_train)

auc_score = roc_auc_score(y_test, logistic_model.predict_proba(X_test)[:, 1])
churn_prob = logistic_model.predict_proba(star_cust[logistic_features])[0, 1]

print_card("🎯 ERREGRESIO LOGISTIKOA", [
    f"Ereduak nor doan eta nor geratzen den bereizteko duen gaitasuna (AUC): {auc_score:.2f}",
    f"Gure izar bezeroak baja emateko duen probabilitatea: {churn_prob * 100:.1f}%",
    "⚠️ % 50etik gora izanda, bezero hau atxikipenerako lehentasunezko kontaktuen zerrendan sartuko litzateke.",
])

---
## 3️⃣ KNN (K Auzokide Hurbilenak) — "Zer gomendatuko diogu?"

Haren antzekoen diren bezeroak bilatzen ditu, eta ikusten du zein produktu-kategoria erosten duten gehien.

👉 **Probatu graduatzailea (slider-a) mugitzen** eta behatu nola aldatzen den gomendioa begiratzen duzun "auzokide" kopuruaren arabera.

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from ipywidgets import interact, IntSlider

knn_features = ["current_monthly_spend", "monthly_purchases", "tenure_months", "days_since_last_purchase"]

scaler_knn = StandardScaler()
X_scaled = scaler_knn.fit_transform(df[knn_features])
star_cust_scaled = scaler_knn.transform(star_cust[knn_features])


def recommend(k=5):
    nn_model = NearestNeighbors(n_neighbors=k)
    nn_model.fit(X_scaled)
    _, idx = nn_model.kneighbors(star_cust_scaled)
    neighbors = df.iloc[idx[0]]
    counts = neighbors["favorite_category"].value_counts()
    print_card(f"🧭 KNN (K={k})", [
        f"Bere {k} auzokide antzekoenen artean gehien erositako kategoria: {counts.idxmax()}",
        f"Banaketa osoa: {dict(counts)}",
    ])


interact(recommend, k=IntSlider(min=3, max=50, step=1, value=5, description="K auzokide"));

---
## 4️⃣ Naive Bayes — "Zer iritzi dute guri buruz?"

Hemen datu-motaz aldatuko dugu: zenbakien ordez, **testua**. Bezeroen iritziak/iruzkinak positibo edo negatibo gisa sailkatuko ditugu.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    reviews.review_text, reviews.positive_sentiment, test_size=0.2, random_state=42
)

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(X_train_txt)
X_test = vectorizer.transform(X_test_txt)

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
acc = accuracy_score(y_test, nb_model.predict(X_test))

# Eskuz idatzitako iruzkin berri batekin probatuko dugu
new_review = ["Eskaera berandu iritsi zen eta laguntza-zerbitzuak ez die nire mezuei erantzuten"]
proba = nb_model.predict_proba(vectorizer.transform(new_review))[0]

print_card("💬 NAIVE BAYES", [
    f"Ereduaren zehaztasuna inoiz ikusi ez dituen iruzkinak sailkatzean: {acc * 100:.1f}%",
    f'Probako iruzkina: "{new_review[0]}"',
    f"Positiboa izateko probabilitatea: {proba[1] * 100:.1f}% | negatiboa: {proba[0] * 100:.1f}%",
])
print("✏️  Aldatu goiko 'new_review' testua eta exekutatu berriro gelaxka zure adibideak probatzeko.")

---
## 5️⃣ K-means — "Zein bezero-taldetan dago?"

Hemen ez dago ereduari eman beharreko erantzun zuzenik (etiketarik) — ereduak berak aurkitzen ditu taldeak (klusterrak).

👉 **Mugitu talde-kopuruaren graduatzailea (slider-a)** eta behatu nola aldatzen diren profilak eta gure bezeroa zein taldetan kokatzen den.

In [ ]:
from sklearn.cluster import KMeans

kmeans_features = ["current_monthly_spend", "monthly_purchases", "days_since_last_purchase", "tenure_months"]
scaler_km = StandardScaler()
X_scaled_km = scaler_km.fit_transform(df[kmeans_features])
star_cust_scaled_km = scaler_km.transform(star_cust[kmeans_features])


def segment_customers(k=4):
    kmeans_model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans_model.fit_predict(X_scaled_km)
    df_tmp = df.copy()
    df_tmp["cluster"] = labels
    summary = df_tmp.groupby("cluster")[kmeans_features].mean().round(1)
    star_cluster = kmeans_model.predict(star_cust_scaled_km)[0]

    print(f"--- Ereduak aurkitutako {k} taldeak ---")
    display(summary)
    print(f"\n👉 Gure izar bezeroa {star_cluster}. taldean dago\n")

    plt.figure(figsize=(6, 4))
    plt.scatter(df_tmp["tenure_months"], df_tmp["current_monthly_spend"],
                c=df_tmp["cluster"], cmap="tab10", alpha=0.4, s=12)
    plt.scatter(star_cust["tenure_months"], star_cust["current_monthly_spend"],
                c="black", marker="*", s=300, label="Izar bezeroa")
    plt.xlabel("Antzinatasuna (hilabeteak)")
    plt.ylabel("Hileko egungo gastua (€)")
    plt.title(f"Bezeroen segmentazioa (K={k})")
    plt.legend()
    plt.show()


interact(segment_customers, k=IntSlider(min=2, max=8, step=1, value=4, description="Talde kopurua"));

---
## 6️⃣ Erabaki-zuhaitza — "Bidali al diogu kanpaina?"

Ikuskatzeko eredurik errazena: literalki egin dituen galderak ikus ditzakezu.

Haren **erregresio-aldaera** ere ikusiko dugu: bai/ez beharrean, zenbaki bat (aurreikusitako batez besteko erosketa-tiketa).

In [ ]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree

tree_features = ["days_since_last_purchase", "monthly_purchases", "tenure_months"]
X_train, X_test, y_train, y_test = train_test_split(
    df[tree_features], df["responded_campaign"], test_size=0.2, random_state=42, stratify=df["responded_campaign"]
)

tree_clf = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_clf.fit(X_train, y_train)
acc_tree = accuracy_score(y_test, tree_clf.predict(X_test))

pred_class = tree_clf.predict(star_cust[tree_features])[0]
proba_tree = tree_clf.predict_proba(star_cust[tree_features])[0]

# Erregresio-aldaera: espero den batez besteko tiketa
tree_reg = DecisionTreeRegressor(max_depth=4, random_state=42)
tree_reg.fit(df[tree_features], df["current_monthly_spend"])
pred_ticket = tree_reg.predict(star_cust[tree_features])[0]

print_card("🌳 ERABAKI-ZUHAITZA", [
    f"Ereduaren zehaztasuna inoiz ikusi ez dituen datuetan: {acc_tree * 100:.1f}%",
    f"Kanpaina bidali? {'BAI' if pred_class == 1 else 'EZ'} (konfiantza: {max(proba_tree) * 100:.1f}%)",
    f"Erregresio-aldaera — espero den batez besteko tiketa: {pred_ticket:.2f}€",
])

plt.figure(figsize=(16, 6))
plot_tree(tree_clf, feature_names=tree_features, class_names=["Ez du erantzuten", "Erantzuten du"],
          filled=True, fontsize=8)
plt.title("Hain literala da erabaki-zuhaitz bat — hatzarekin jarrai dezakezu bidea")
plt.show()

---
## 7️⃣ Random Forest (Ausazko Basoa) — "Ez fidatu zuhaitz bakar batez"

Banakako 5 zuhaitz entrenatuko ditugu, bakoitza datuen lagin desberdin batekin, eta ikusiko dugu gure bezeroaren inguruan ados jartzen diren ala ez. Ondoren, elkarrekin bozkatzen duten 300 zuhaitzeko baso batekin konparatuko dugu.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rng_local = np.random.default_rng(0)
individual_predictions = []
print("Banakako 5 zuhaitz, bakoitza lagin desberdin batekin entrenatua:\n")
for i in range(5):
    idx = rng_local.choice(len(X_train), size=len(X_train), replace=True)
    Xb, yb = X_train.iloc[idx], y_train.iloc[idx]
    tree_single = DecisionTreeClassifier(max_depth=4, random_state=i)
    tree_single.fit(Xb, yb)
    p = tree_single.predict(star_cust[tree_features])[0]
    individual_predictions.append(p)
    print(f"  {i+1}. zuhaitza: {'BAI bidali' if p == 1 else 'EZ bidali'}")

num_yes = individual_predictions.count(1)
print(f"\n👉 5 zuhaitzetatik {num_yes}k 'bai' diote, {5 - num_yes}k 'ez' — hain ezegonkorra izan daiteke zuhaitz bakar bat.\n")

forest_model = RandomForestClassifier(n_estimators=300, max_depth=4, random_state=42)
forest_model.fit(X_train, y_train)
acc_forest = accuracy_score(y_test, forest_model.predict(X_test))
proba_forest = forest_model.predict_proba(star_cust[tree_features])[0]

print_card("🌲🌲🌲 RANDOM FOREST (300 zuhaitz)", [
    f"Ereduaren zehaztasuna inoiz ikusi ez dituen datuetan: {acc_forest * 100:.1f}%",
    f"Kanpainari erantzuteko probabilitatea, basoaren adostasunaren arabera: {proba_forest[1] * 100:.1f}%",
])

---
## 8️⃣ XGBoost — "Errore-euro bakoitzak balio duenean"

1. blokeko galderara itzuliko gara (zenbat gastatuko du?), baina orain zuhaitzez zuhaitz bere akatsak zuzentzen dituen eredu batekin. Benetako errorea erregresio linealarenarekin alderatuko dugu.

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

xgb_features = ["current_monthly_spend", "tenure_months", "monthly_purchases",
                "days_since_last_purchase", "monthly_email_opens", "support_ticket"]
X_train, X_test, y_train, y_test = train_test_split(
    df[xgb_features], df["next_monthly_spend"], test_size=0.2, random_state=42
)

# Erregresio lineala, baldintza berdinetan alderatzeko
linear_benchmark = LinearRegression().fit(X_train, y_train)
mae_linear = mean_absolute_error(y_test, linear_benchmark.predict(X_test))

xgb_model = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42)
xgb_model.fit(X_train, y_train)
mae_xgb = mean_absolute_error(y_test, xgb_model.predict(X_test))

pred_xgb = xgb_model.predict(star_cust[xgb_features])[0]
pred_linear_comp = linear_benchmark.predict(star_cust[xgb_features])[0]

print_card("🚀 XGBOOST vs ERREGRESIO LINEALA", [
    f"Batez besteko errorea (MAE) Erregresio Lineala: {mae_linear:.2f}€",
    f"Batez besteko errorea (MAE) XGBoost:            {mae_xgb:.2f}€",
    f"XGBoost-en hobekuntza: % {(1 - mae_xgb / mae_linear) * 100:.1f} errore gutxiago",
    f"Erregresio linealaren aurreikuspena izar bezeroarentzat: {pred_linear_comp:.2f}€",
    f"XGBoost-en aurreikuspena izar bezeroarentzat:           {pred_xgb:.2f}€",
])
print("Desberdintasuna erregresio linealak harrapatu ezin dituen erlazioetatik dator")
print("(atalaseak, aldagaien konbinazioak), XGBoost-ek bere kabuz hautematen dituenak.")

---
## 9️⃣ LightGBM — "Datu-bolumena asko handitzen denean"

XGBoost-en ataza bera, baina orain **300.000 bezerorekin** 2.000 beharrean, bakoitzaren abiadura proban jartzeko. Ikusiko dituzun denborak zeure exekuzioarenak dira, ez asmatutako zenbakiak.

In [ ]:
import time
from lightgbm import LGBMRegressor

print("300.000 bezeroko datu-multzoa sortzen abiadura-probarako...")
df_large = generate_dataset(n=300_000, seed=123)
X_large = df_large[xgb_features]
y_large = df_large["next_monthly_spend"]

t0 = time.time()
XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, random_state=42).fit(X_large, y_large)
t_xgb = time.time() - t0

t0 = time.time()
LGBMRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, random_state=42, verbose=-1).fit(X_large, y_large)
t_lgbm = time.time() - t0

print_card("⚡ LIGHTGBM vs XGBOOST — 300.000 errenkada", [
    f"XGBoost entrenamendu-denbora:  {t_xgb:.2f} segundo",
    f"LightGBM entrenamendu-denbora: {t_lgbm:.2f} segundo",
])
if t_lgbm < t_xgb:
    print(f"Exekuzio honetan, LightGBM {t_xgb / t_lgbm:.2f}x azkarragoa izan da.")
else:
    print(f"Exekuzio honetan, biak oso pareko ibili dira ({t_xgb:.1f}s vs {t_lgbm:.1f}s).")
print("\nOhar zintzoa: XGBoost-en bertsio modernoek asko murriztu dute abiadura-aldea.")
print("LightGBM-ren benetako abantaila gaur egun memorian eta are bolumen handiagoetako berritrenamendu maizetan nabaritzen da gehienbat.")

---
## 🗂️ Amaiera — Gure bezeroaren fitxa osoa, 9 ereduek eraikia

Ez dago eredu "onena" modu orokorrean — bakoitzak bezero beraren inguruko negozio-galdera desberdin bati erantzuten dio. Hemen dituzu 9 erantzunak batera.

In [ ]:
print("=" * 58)
print(f"FITXA OSOA — #{star_cust.customer_id.iloc[0]} bezeroa")
print("=" * 58)
print(f" 1. Erregresio Lineala    → ~{pred_linear:.0f}€ gastatuko ditu hurrengo hilabetean")
print(f" 2. Erregresio Logistikoa → % {churn_prob * 100:.0f}-ko baja-probabilitatea")
print(f" 3. KNN                   → bere auzokideek gehien ikusitako kategoriako produktuak gomendatu")
print(f" 4. Naive Bayes           → (testu-iruzkinei aplikatua, baldin balitu)")
print(f" 5. K-means               → portaera-talde jakin batekoa da (ikus goian)")
print(f" 6. Erabaki-zuhaitza      → kanpaina bidali? {'BAI' if pred_class == 1 else 'EZ'} (espero den batez besteko tiketa: {pred_ticket:.0f}€)")
print(f" 7. Random Forest         → % {proba_forest[1] * 100:.0f}-ko erantzun-probabilitatea (300 zuhaitzen adostasuna)")
print(f" 8. XGBoost               → ~{pred_xgb:.0f}€ gastatuko ditu hurrengo hilabetean (aurreikuspen zehatzagoa)")
print(f" 9. LightGBM              → XGBoost-en antzeko aurreikuspena, milioika bezerotara eskalatzeko pentsatua")
print("=" * 58)
print("\nEredu bakoitzak angelu desberdin batetik begiratzen dio bezero berari.")
print("Galdera zuzena ez da 'zein da eredurik onena?', baizik eta 'zer negozio-galdera erantzun nahi dut?'")

---
## 🎬 Eta orain zer?

Bideoko 9 ereduak exekutatu dituzu, unean bertan sortzen diren datuen gainean — nahi duzunean itzul zaitezke eta berriro exekutatu, izar bezeroaren zenbakiak aldatu, edo graduatzaileak (slider-ak) ukitu emaitza bakoitza nola aldatzen den ikusteko.

**Gogoratu:** eredu hauek guztiek taula-datuen (datu tabularren) gainean lan egiten zuten. Datua irudi bat, audio bat edo benetako egituratu gabeko testu bat denean, sare neuronalen esparruan sartzen gara.

👀  AZTERKETARAKO GERTU?
